# باب 7: چیٹ ایپلیکیشنز بنانا
## Azure OpenAI API کوئیک اسٹارٹ


## جائزہ
یہ نوٹ بک [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) سے مطابقت رکھتی ہے جس میں نوٹ بکس شامل ہیں جو [OpenAI](notebook-openai.ipynb) سروس تک بھی رسائی حاصل کرتی ہیں۔

Python OpenAI API Azure OpenAI کے ساتھ بھی کام کرتا ہے، چند تبدیلیوں کے ساتھ۔ یہاں فرق کے بارے میں مزید جانیں: [Python کے ساتھ OpenAI اور Azure OpenAI اینڈ پوائنٹس کے درمیان کیسے سوئچ کریں](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

مزید ابتدائی مثالوں کے لیے براہ کرم سرکاری [Azure OpenAI Quickstart Documentation](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst) دیکھیں۔


## فہرست مضامین  

[جائزہ](#overview)  
[Azure OpenAI سروس کے ساتھ شروع کریں](#getting-started-with-azure-openai-service)  
[اپنا پہلا پرامپٹ بنائیں](#build-your-first-prompt)  

[استعمال کے کیسز](#use-cases)    
[1. متن کا خلاصہ کریں](#summarize-text)  
[2. متن کی درجہ بندی کریں](#classify-text)  
[3. نئے پروڈکٹ نام بنائیں](#generate-new-product-names)  
[4. درجہ بندی کرنے والے کو بہتر بنائیں](#fine-tune-a-classifier)  
[5. ایمبیڈنگز](#embeddings)

[حوالہ جات](#references)


### ایزور اوپن اے آئی سروس کے ساتھ آغاز کرنا

نئے گاہکوں کو ایزور اوپن اے آئی سروس کے لیے [رسائی کے لیے درخواست دینی ہوگی](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst)۔  
منظوری مکمل ہونے کے بعد، گاہک ایزور پورٹل میں لاگ ان کر سکتے ہیں، ایزور اوپن اے آئی سروس کا ایک ریسورس بنا سکتے ہیں، اور اسٹوڈیو کے ذریعے ماڈلز کے ساتھ تجربہ کرنا شروع کر سکتے ہیں  

[تیزی سے شروعات کرنے کے لیے بہترین وسیلہ](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### اپنا پہلا پرامپٹ بنائیں  
یہ مختصر مشق ایک سادہ کام "خلاصہ سازی" کے لیے OpenAI ماڈل کو پرامپٹس جمع کروانے کے لیے بنیادی تعارف فراہم کرے گی۔


**اقدامات**:  
1. اپنے پائتھون ماحول میں OpenAI لائبریری انسٹال کریں  
2. معیاری مددگار لائبریریز لوڈ کریں اور OpenAI سروس کے لیے اپنے معمول کے OpenAI سیکیورٹی اسناد مرتب کریں جو آپ نے بنایا ہے  
3. اپنے کام کے لیے ایک ماڈل کا انتخاب کریں  
4. ماڈل کے لیے ایک سادہ پرامپٹ بنائیں  
5. اپنی درخواست ماڈل API کو جمع کروائیں!


### 1. اوپن اے آئی انسٹال کریں


  > [!NOTE] یہ قدم ضروری نہیں ہے اگر یہ نوٹ بک Codespaces پر یا Devcontainer کے اندر چلائی جائے۔


In [ ]:
%pip install openai python-dotenv

### 2. معاون لائبریریاں درآمد کریں اور سندات کا آغاز کریں


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. صحیح ماڈل تلاش کرنا  
ماڈلز جیسے کہ GPT-4o اور GPT-4o منی قدرتی زبان کو سمجھ سکتے ہیں اور اس میں پیداوار کر سکتے ہیں۔ مائیکروسافٹ فاؤنڈری میں Azure OpenAI متعدد ماڈل کی صلاحیتیں پیش کرتا ہے، ہر ایک مختلف طاقت اور رفتار کی سطح کے ساتھ جو مختلف کاموں کے لیے موزوں ہے۔ 

[مائیکروسافٹ فاؤنڈری میں Azure OpenAI کے ماڈلز](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## 4. پرامپٹ ڈیزائن  

"بڑی زبان کے ماڈلز کا جادو یہ ہے کہ وسیع مقدار میں متن پر اس پیش گوئی کی غلطی کو کم سے کم کرنے کی تربیت دی جاتی ہے، جس سے ماڈلز ان پیش گوئیوں کے لیے مفید تصورات سیکھتے ہیں۔ مثال کے طور پر، وہ ایسے تصورات سیکھتے ہیں"(1):

* ہجے کرنے کا طریقہ
* گرامر کیسے کام کرتی ہے
* موازنہ کرنے کا طریقہ
* سوالات کے جواب دینے کا طریقہ
* بات چیت رکھنے کا طریقہ
* کئی زبانوں میں لکھنے کا طریقہ
* کوڈنگ کا طریقہ
* وغیرہ۔

#### ایک بڑے زبان کے ماڈل کو کنٹرول کرنے کا طریقہ  
"بڑے زبان کے ماڈل میں تمام ان پٹس میں، سب سے زیادہ اثرانداز کرنے والا عنصر ٹیکسٹ پرامپٹ ہے(1)۔

بڑے زبان کے ماڈلز کو چند طریقوں سے آؤٹ پٹ پیدا کرنے کے لیے پرامپٹ کیا جا سکتا ہے:

- ہدایت: ماڈل کو بتائیں کہ آپ کیا چاہتے ہیں
- تکمیل: ماڈل کو وہ مکمل کرنے پر آمادہ کریں جو آپ شروع کر چکے ہیں
- مظاہرہ: ماڈل کو دکھائیں کہ آپ کیا چاہتے ہیں، درج ذیل میں سے کسی کے ساتھ:
- پرامپٹ میں چند مثالیں
- ایک فائن-ٹیوننگ ٹریننگ ڈیٹا سیٹ میں سینکڑوں یا ہزاروں مثالیں



#### پرامپٹ بنانے کے تین بنیادی اصول ہیں:

**دکھائیں اور بتائیں**۔ واضح کریں کہ آپ کیا چاہتے ہیں، چاہے وہ ہدایات کے ذریعے ہو، مثالوں کے ذریعے ہو، یا دونوں کا امتزاج ہو۔ اگر آپ چاہتے ہیں کہ ماڈل کسی اشیاء کی فہرست کو حروف تہجی کی ترتیب میں درجہ بندی کرے یا پیراگراف کو جذبات کی بنیاد پر درجہ بند کرے، تو اسے دکھائیں کہ یہی آپ کی خواہش ہے۔

**معیاری ڈیٹا فراہم کریں**۔ اگر آپ ایک کلاسیفائر بنانے کی کوشش کر رہے ہیں یا ماڈل کو کسی نمونے کی پیروی کرانا چاہتے ہیں، تو یقینی بنائیں کہ کافی مثالیں موجود ہوں۔ اپنی مثالوں کو پروف ریڈ کریں — ماڈل عام طور پر بنیادی ہجے کی غلطیوں کو سمجھ سکتا ہے اور آپ کو جواب دے سکتا ہے، مگر ہو سکتا ہے کہ وہ اسے جان بوجھ کر سمجھے اور اس کا اثر جواب پر پڑے۔

**اپنی ترتیبات چیک کریں۔** درجہ حرارت اور top_p کی ترتیبات یہ کنٹرول کرتی ہیں کہ ماڈل جواب دینے میں کتنی یقینی ہو۔ اگر آپ ایسا جواب چاہتے ہیں جس کا صرف ایک صحیح جواب ہو، تو آپ انہیں کم سیٹ کرنا چاہیں گے۔ اگر آپ زیادہ متنوع جوابات چاہتے ہیں، تو آپ انہیں زیادہ سیٹ کر سکتے ہیں۔ اس سلسلے میں سب سے بڑی غلطی لوگوں کی یہ ہوتی ہے کہ وہ ان ترتیبات کو "چالاکی" یا "تخلیقی صلاحیت" کے کنٹرول سمجھ لیتے ہیں۔


ماخذ: https://learn.microsoft.com/azure/ai-foundry/openai/overview


تصویر آپ کا پہلا متن پرامپٹ بنا رہی ہے!


### 5. جمع کروائیں!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### ایک ہی کال دہرائیں، نتائج کا موازنہ کیسے ہوتا ہے؟


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## متن کا خلاصہ کریں  
#### چیلنج  
متن کے آخر میں 'tl;dr:' شامل کرکے متن کا خلاصہ کریں۔ نوٹ کریں کہ ماڈل بغیر اضافی ہدایات کے کئی کام انجام دینے کا طریقہ کیسے سمجھتا ہے۔ آپ ماڈل کے رویے میں تبدیلی اور موصول ہونے والے خلاصے کو حسب ضرورت بنانے کے لیے tl;dr سے زیادہ وضاحتی پرامپٹس کے ساتھ تجربہ کر سکتے ہیں(3)۔  

حالیہ کام نے ایک بڑے متن کے مجموعے پر پری ٹریننگ کے بعد مخصوص کام پر فائن ٹوننگ کرکے کئی NLP کاموں اور بینچ مارکس پر نمایاں بہتری دکھائی ہے۔ اگرچہ عام طور پر فن تعمیر میں کام سے آزاد ہوتا ہے، یہ طریقہ اب بھی ہزاروں یا کئی ہزار مثالوں کے مخصوص کام کے فائن ٹوننگ ڈیٹا سیٹس کا تقاضا کرتا ہے۔ اس کے برعکس، انسان عام طور پر چند مثالوں یا آسان ہدایات سے نیا زبان کا کام انجام دے سکتے ہیں - ایسا کچھ جو موجودہ NLP نظام اب بھی بڑی حد تک کرنے میں مشکل پیش آتی ہے۔ یہاں ہم دکھاتے ہیں کہ زبان کے ماڈلز کو بڑھانا کام سے آزاد، چند شاٹس کی کارکردگی کو بہت بہتر بناتا ہے، بعض اوقات یہاں تک کہ پچھلے ریاستی فن تعمیر کے فائن ٹوننگ طریقوں کے مقابلہ میں بھی پہنچ جاتا ہے۔  



خلاصہ  


# متعدد استعمال کے کیسز کے لیے مشقیں  
1. متن کا خلاصہ بنائیں  
2. متن کی درجہ بندی کریں  
3. نئے مصنوعات کے نام تیار کریں
4. ایمبیڈنگز
5. ایک کلاسفائر کو فائن ٹیون کریں


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## متن کو درجہ بندی کریں  
#### چیلنج  
آئیٹمز کو اس وقت فراہم کردہ زمروں میں درجہ بند کریں جب انفرینس ہو رہا ہو۔ درج ذیل مثال میں، ہم زمرے اور درجہ بندی کے لیے متن دونوں کو پرامپٹ میں فراہم کرتے ہیں(*playground_reference). 

صارف کی استفسار: ہیلو، میرے لیپ ٹاپ کے کی بورڈ کی ایک چابی حال ہی میں ٹوٹ گئی ہے اور مجھے اس کا متبادل چاہیے:  

درجہ بند شدہ زمرہ:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## نئے پروڈکٹ کے نام بنائیں
#### چیلنج
مثال کے الفاظ سے پروڈکٹ کے نام بنائیں۔ یہاں ہم پرامپٹ میں اس پروڈکٹ کی معلومات شامل کرتے ہیں جس کے لیے ہم نام تیار کرنے جا رہے ہیں۔ ہم ایک مشابہ مثال بھی فراہم کرتے ہیں تاکہ وہ پیٹرن دکھایا جا سکے جو ہم چاہتے ہیں۔ ہم نے درجہ حرارت کی قیمت بھی زیادہ رکھی ہے تاکہ زیادہ تصادفی اور جدید جوابات حاصل ہوں۔

پروڈکٹ کی وضاحت: گھر کا ملک شیک تیار کرنے والا آلہ
شروع کے الفاظ: تیز، صحت مند، کمپیکٹ۔
پروڈکٹ کے نام: ہوم شیکر، فٹ شیکر، کوئیک شیک، شیک میکر

پروڈکٹ کی وضاحت: جوتے کا ایک جوڑا جو کسی بھی جوتے کے سائز میں فٹ ہو سکتا ہے۔
شروع کے الفاظ: قابل تطابق، فٹ، اومنی فٹ۔


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## امبیڈنگز
اس سیکشن میں دکھایا جائے گا کہ امبیڈنگز کیسے حاصل کی جائیں، اور الفاظ، جملوں، اور دستاویزات کے مابین مشابہت کیسے تلاش کی جائے۔ درج ذیل نوٹ بکس چلانے کے لیے آپ کو ایک ایسا ماڈل تعینات کرنا ہوگا جو `text-embedding-ada-002` کو بنیادی ماڈل کے طور پر استعمال کرتا ہو اور اس کی تعیناتی کا نام .env فائل میں، `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` متغیر کے ذریعے سیٹ کرنا ہوگا۔


### ماڈل درجہ بندی - ایک ایمبیڈنگ ماڈل کا انتخاب  

**ماڈل درجہ بندی**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> text-embedding  (ایمبیڈنگز خاندان)  
{capability} --> ada             (تمام دیگر ایمبیڈنگ ماڈلز 2024 میں بند کر دیے جائیں گے)  
{input-type} --> n/a             (صرف سرچ ماڈلز کے لیے مخصوص)  
{identifier} --> 002             (ورژن 002)  

model = 'text-embedding-ada-002'


  > [!NOTE] اگر یہ نوٹ بک Codespaces پر یا Devcontainer کے اندر چلائی جائے تو درج ذیل قدم ضروری نہیں ہے


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## سی این این ڈیلی نیوز ڈیٹا سیٹ سے مضمون کا موازنہ
source: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# حوالہ جات  
- [Microsoft Foundry - Azure OpenAI Models](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# مزید مدد کے لئے  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com)


# حصہ ڈالنے والے
* لوئس لی  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
